# CRVSE MCD transfer-learning triage

One controlled experiment to rank which vitals are learnable from the froze **PhysFormer-multichannel** HR backbone, on the only dataset that carries every label (MCD-rPPG).



## 1. Config and Setup

In [1]:
import h5py, os, torch, optuna, json, math
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from collections import defaultdict
from scipy import stats
from scipy.signal import resample
from sklearn.linear_model import LinearRegression

H5_PATH= "/kaggle/input/datasets/cezarytubacki/mcd-rppg-ens/mcd_rppg_ensemble.h5"
CKPT_PATH = "/kaggle/input/notebooks/cezarytubacki/crvse-ensemble-optuna-trials-transformer-multichan/CRVSETransformer_Ensemble_transformer_multichannel_best.pt"
SQI_GATE = 0.10

HEADS = {
    "RR" : (["bm_respiratory_rate"], [(6.0, 40.0)]),
    "BP" : (["bm_systolic_bp", "bm_diastolic_bp"], [(70.0, 220.0), (40.0, 130.0)]),
    "SpO2" : (["bm_spo2"], [(85.0, 100.0)]),
    "Stiffness" : (["bm_arterial_stiffness"], [(3.0, 18.0)]),
}
# Demographic baseline inputs (for the BP confound test).
DEMO_KEYS = ["bm_age", "bm_sex", "bm_bmi"]

TARGET_FRAMES, WINDOW_SEC, STRIDE_SEC, SEED = 240, 8, 4, 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
N_TRIALS_HEAD = 15 
HEAD_EPOCHS = 60
torch.manual_seed(SEED); np.random.seed(SEED)
optuna.logging.set_verbosity(optuna.logging.WARNING)
print("MCD triage |", "cuda" if torch.cuda.is_available() else "cpu", "| heads:", list(HEADS))

MCD triage | cuda | heads: ['RR', 'BP', 'SpO2', 'Stiffness']


## 2. Windows

In [2]:
def extract_windows(rec, fps):
    """Return list of dicts: signal (3,240) + every label/demographic value (scalar, broadcast)."""
    wlen, wstr = int(fps * WINDOW_SEC), int(fps * STRIDE_SEC)
    n = len(rec["rppg_pos"])
    # read all candidate scalar labels + demographics once per recording
    vals = {}
    for keys, _ in HEADS.values():
        for k in keys: vals[k] = float(rec.attrs[k]) if k in rec.attrs else np.nan
    
    def _to_float(v):
        if isinstance(v, bytes):
            v = v.decode()
        if isinstance(v, str):
            s = v.strip().lower()
            return {"m": 1.0, "male": 1.0, "f": 0.0, "female": 0.0}.get(s, np.nan)
        return float(v)
        
    for k in DEMO_KEYS:
        vals[k] = _to_float(rec.attrs[k]) if k in rec.attrs else np.nan
        
    sig = {c: rec[f"rppg_{c}"][:] for c in ("pos","chrom","green")}
    out = []
    for st in range(0, n-wlen+1, wstr):
        chans, bad = [], False
        for c in ("pos","chrom","green"):
            s = sig[c][st:st+wlen]
            if np.any(np.isnan(s)): 
                bad=True; break
            if len(s)!=TARGET_FRAMES: 
                s = resample(s, TARGET_FRAMES).astype(np.float32)
            if s.std()<1e-6: 
                bad=True; break
            chans.append(((s-s.mean())/s.std()).astype(np.float32))
        if bad: 
            continue
        out.append({"signal": np.stack(chans), **vals})
    return out

In [3]:
def build_index(path):
    samples = []
    with h5py.File(path,"r") as f:
        for sid in f["subjects"].keys():
            for rid in f["subjects"][sid]["recordings"].keys():
                rec = f["subjects"][sid]["recordings"][rid]
                if float(rec.attrs.get("sqi_ensemble",1.0)) < SQI_GATE: 
                    continue
                for w in extract_windows(rec, float(rec.attrs["fps"])):
                    w["subject"] = f"mcd__{sid}"; w["rec"] = f"{sid}/{rid}"
                    samples.append(w)
    print(f"windows: {len(samples)} | subjects: {len(set(s['subject'] for s in samples))}")
    return samples

In [4]:
all_samples = build_index(H5_PATH)

windows: 11368 | subjects: 203


## 3. Model Architecture

In [9]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 300, dropout: float = 0.1) -> None:
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)

class TransformerEncoderLayerCustom(nn.Module):
    def __init__(self, d_model: int, n_heads: int, dim_feedforward: int = 256, dropout: float = 0.1) -> None:
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.scale = self.head_dim ** -0.5
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.ff = nn.Sequential(
            nn.Linear(d_model, dim_feedforward),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(dim_feedforward, d_model),
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
    def _attention(self, x: torch.Tensor) -> torch.Tensor:
        B, T, _ = x.shape
        Q = self.q_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        K = self.k_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        V = self.v_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        scores = torch.matmul(Q, K.transpose(-2, -1)) * self.scale
        attn = F.softmax(scores, dim=-1)
        attn = self.dropout(attn)
        out = torch.matmul(attn, V)
        out = out.transpose(1, 2).contiguous().view(B, T, -1)
        return self.out_proj(out)
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.dropout(self._attention(self.norm1(x)))
        x = x + self.dropout(self.ff(self.norm2(x)))
        return x

class CRVSETransformer(nn.Module):
    def __init__(self, in_channels: int = 1, cnn_channels: int = 64, n_heads: int = 4, n_layers: int = 2,
                 dim_feedforward: int = 256, dropout: float = 0.2, hr_min: float = 40.0, hr_max: float = 180.0) -> None:
        super().__init__()
        assert cnn_channels % n_heads == 0
        self.hr_min = hr_min
        self.hr_max = hr_max
        self.cnn_channels = cnn_channels
        self.encoder = nn.Sequential(
            nn.Conv1d(in_channels, cnn_channels // 2, kernel_size=7, padding=3),
            nn.BatchNorm1d(cnn_channels // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Conv1d(cnn_channels // 2, cnn_channels, kernel_size=5, padding=2),
            nn.BatchNorm1d(cnn_channels),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Conv1d(cnn_channels, cnn_channels, kernel_size=3, padding=1),
            nn.BatchNorm1d(cnn_channels),
            nn.ReLU(),
            nn.Dropout(dropout),
        )
        self.pos_enc = PositionalEncoding(cnn_channels, max_len=300, dropout=dropout)
        self.transformer_layers = nn.ModuleList([
            TransformerEncoderLayerCustom(
                d_model = cnn_channels,
                n_heads = n_heads,
                dim_feedforward = dim_feedforward,
                dropout = dropout,
            )
            for _ in range(n_layers)
        ])
        self.head = nn.Sequential(
            nn.Linear(cnn_channels, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 1),
        )
    def extract_features(self, x: torch.Tensor) -> torch.Tensor:
        """Extract feature vector (B, d_model) WITHOUT the regression head."""
        out = self.encoder(x)
        out = out.permute(0, 2, 1)
        out = self.pos_enc(out)
        for layer in self.transformer_layers:
            out = layer(out)
        out = out.mean(dim=1)
        return out
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Full forward pass with HR prediction."""
        out = self.extract_features(x)
        out = self.head(out).squeeze(-1)
        if not self.training:
            out = out.clamp(self.hr_min, self.hr_max)
        return out

## 4. Load Frozen Backbone

In [10]:
def load_backbone(ckpt_path):
    """Load backbone - handles both dim_feedforward and ff_mult formats."""
    ckpt = torch.load(ckpt_path, map_location="cpu")
    params = ckpt["best_params"]
    
    # Handle dim_feedforward vs ff_mult
    if "dim_feedforward" in params:
        dim_feedforward = params["dim_feedforward"]
    elif "ff_mult" in params:
        dim_feedforward = int(params["cnn_channels"] * params["ff_mult"])
    else:
        dim_feedforward = int(params["cnn_channels"] * 4.0)
    
    backbone = CRVSETransformer(
        in_channels=ckpt.get("in_channels", 3),
        cnn_channels=params["cnn_channels"],
        n_heads=params["n_heads"],
        n_layers=params["n_layers"],
        dim_feedforward=dim_feedforward,
        dropout=params.get("dropout", 0.2),
    )
    own_state = backbone.state_dict()
    ckpt_state = ckpt["model_state"]
    kept = {k: v for k, v in ckpt_state.items() if k in own_state and own_state[k].shape == v.shape}
    own_state.update(kept)
    backbone.load_state_dict(own_state)
    for param in backbone.parameters():
        param.requires_grad = False
    backbone.eval().to(DEVICE)
    print(f"Loaded {len(kept)}/{len(ckpt_state)} tensors | d_model={params['cnn_channels']} | FROZEN")
    return backbone, params["cnn_channels"]

backbone, d_model = load_backbone(CKPT_PATH)
print(f"d_model: {d_model}")

Loaded 39/42 tensors | d_model=128 | FROZEN
d_model: 128


## 5. Extract Frozen Features

In [11]:
# Filter samples with SQI
valid_idx = [i for i, s in enumerate(all_samples) if 'signal' in s]
print(f"Valid samples: {len(valid_idx)}")

# Extract features
FEATURES = np.zeros((len(valid_idx), d_model), dtype=np.float32)
SUBJECTS = np.array([all_samples[i]['subject'] for i in valid_idx])

batch_size = 32
backbone.eval()
with torch.inference_mode():
    for start in range(0, len(valid_idx), batch_size):
        end = min(start + batch_size, len(valid_idx))
        batch = [all_samples[valid_idx[i]]['signal'] for i in range(start, end)]
        batch = torch.tensor(np.stack(batch), dtype=torch.float32).to(DEVICE)
        feats = backbone.extract_features(batch).cpu().numpy()
        FEATURES[start:end] = feats

print(f"Features extracted: {FEATURES.shape}")

Valid samples: 11368
Features extracted: (11368, 128)


## 6. Subject Split

In [12]:
def subject_split(subject_array, train_frac=0.7, val_frac=0.15, seed=42):
    rng = np.random.default_rng(seed)
    unique_subjects = np.unique(subject_array)
    rng.shuffle(unique_subjects)
    n = len(unique_subjects)
    n_train = int(n * train_frac)
    n_val = int(n * val_frac)
    train_set = set(unique_subjects[:n_train])
    val_set = set(unique_subjects[n_train:n_train+n_val])
    test_set = set(unique_subjects[n_train+n_val:])
    TR = np.array([i for i, s in enumerate(subject_array) if s in train_set])
    VA = np.array([i for i, s in enumerate(subject_array) if s in val_set])
    TE = np.array([i for i, s in enumerate(subject_array) if s in test_set])
    print(f"Train: {len(TR)} | Val: {len(VA)} | Test: {len(TE)}")
    return TR, VA, TE

TR, VA, TE = subject_split(SUBJECTS)

Train: 7816 | Val: 1710 | Test: 1842


## 7. Per-Head Silent Optuna Sweep

In [13]:
class MLPHead(nn.Module):
    def __init__(self, d_in, hidden, d_out, dropout):
        super().__init__()
        self.net=nn.Sequential(nn.Linear(d_in,hidden),nn.ReLU(),nn.Dropout(dropout),nn.Linear(hidden,d_out))
    def forward(self,x): return self.net(x)

def _valid_mask(keys, ranges):
    m=np.ones(len(all_samples),dtype=bool)
    Y=np.zeros((len(all_samples),len(keys)),dtype=np.float32)
    for j,(k,(lo,hi)) in enumerate(zip(keys,ranges)):
        col=np.array([s.get(k,np.nan) for s in all_samples],dtype=np.float32)
        Y[:,j]=col; m &= np.isfinite(col) & (col>=lo) & (col<=hi)
    return m, Y

def _train_head(Xtr,Ytr,Xva,Yva,hidden,lr,wd,dropout,d_out):
    model=MLPHead(Xtr.shape[1],hidden,d_out,dropout).to(DEVICE)
    opt=AdamW(model.parameters(),lr=lr,weight_decay=wd); lossf=nn.HuberLoss(delta=3.0)
    xt=torch.tensor(Xtr,device=DEVICE); yt=torch.tensor(Ytr,device=DEVICE)
    xv=torch.tensor(Xva,device=DEVICE); yv=torch.tensor(Yva,device=DEVICE)
    best=float("inf")
    for _ in range(HEAD_EPOCHS):
        model.train(); opt.zero_grad(); loss=lossf(model(xt),yt); loss.backward(); opt.step()
        model.eval()
        with torch.inference_mode(): v=torch.mean(torch.abs(model(xv)-yv)).item()
        best=min(best,v)
    return model, best

def _metrics(pred, true, subj):
    diff=pred-true
    win_mae=float(np.mean(np.abs(diff)))
    sp,sl=defaultdict(list),defaultdict(list)
    for p,t,s in zip(pred,true,subj): sp[s].append(p); sl[s].append(t)
    smean_p=np.array([np.mean(sp[k]) for k in sp]); smean_t=np.array([np.mean(sl[k]) for k in sp])
    subj_mae=float(np.mean(np.abs(smean_p-smean_t)))
    wp=np.array(pred)-np.array([np.mean(sp[s]) for s in subj])
    wt=np.array(true)-np.array([np.mean(sl[s]) for s in subj])
    within_mae=float(np.mean(np.abs(wp-wt)))
    r=float(stats.pearsonr(pred,true)[0]) if len(set(true))>1 else float("nan")
    return win_mae, subj_mae, within_mae, r

def run_head(name, keys, ranges):
    m,Y=_valid_mask(keys,ranges); d_out=len(keys)
    tr=TR[m[TR]]; va=VA[m[VA]]; te=TE[m[TE]]
    if min(len(tr),len(va),len(te))<10:
        return {"target":name,"status":f"too few labelled (tr/va/te={len(tr)}/{len(va)}/{len(te)})"}
    Xtr,Ytr=FEATURES[tr],Y[tr]; Xva,Yva=FEATURES[va],Y[va]; Xte,Yte=FEATURES[te],Y[te]
    subj_te=SUBJECTS[te]
    mu=Ytr.mean(0); sd=Ytr.std(0)+1e-6
    Ytr_s, Yva_s = (Ytr-mu)/sd, (Yva-mu)/sd
    def objective(trial):
        hidden=trial.suggest_categorical("hidden",[16,32,64])
        lr=trial.suggest_float("lr",1e-4,5e-3,log=True)
        wd=trial.suggest_float("wd",1e-6,1e-3,log=True)
        dr=trial.suggest_float("dropout",0.0,0.4)
        _,best=_train_head(Xtr,Ytr_s,Xva,Yva_s,hidden,lr,wd,dr,d_out)
        return best
    study=optuna.create_study(direction="minimize",sampler=optuna.samplers.TPESampler(seed=SEED))
    study.optimize(objective,n_trials=N_TRIALS_HEAD,show_progress_bar=False)
    bp=study.best_params
    model,_=_train_head(np.vstack([Xtr,Xva]),np.vstack([Ytr_s,Yva_s]),Xva,Yva_s,
                        bp["hidden"],bp["lr"],bp["wd"],bp["dropout"],d_out)
    model.eval()
    with torch.inference_mode():
        pred=model(torch.tensor(Xte,device=DEVICE)).cpu().numpy()*sd + mu
    rows=[]
    for j,k in enumerate(keys):
        comp=name if d_out==1 else k.replace("bm_","")
        p,t=pred[:,j],Yte[:,j]
        win,subj,within,r=_metrics(p,t,subj_te)
        base=float(np.mean(np.abs(t-Ytr[:,j].mean())))
        row={"target":comp,"win_mae":win,"subj_mae":subj,"within_mae":within,"r":r,
             "baseline_mae":base,"beats_base":base-win,"n_test":len(t)}
        if name=="BP":
            dm,D=_valid_mask(DEMO_KEYS,[(0,200)]*len(DEMO_KEYS))
            tr2=TR[m[TR]&dm[TR]]; te2=TE[m[TE]&dm[TE]]
            if len(tr2)>20 and len(te2)>5:
                lr_d=LinearRegression().fit(D[tr2],Y[tr2,j])
                dmae=float(np.mean(np.abs(lr_d.predict(D[te2])-Y[te2,j])))
                row["demo_base_mae"]=dmae; row["beats_demo"]=dmae-win
        rows.append(row)
    return {"target":name,"rows":rows,"best_params":bp}

results=[run_head(n,k,r) for n,(k,r) in HEADS.items()]

In [14]:
flat=[]
for res in results:
    if "rows" not in res: print(f"[skip] {res['target']}: {res['status']}"); continue
    flat.extend(res["rows"])
flat.sort(key=lambda r: -r["beats_base"])

hdr=f"{'target':<12}{'win MAE':>9}{'subj MAE':>10}{'within':>9}{'r':>7}{'base':>8}{'beats':>8}{'demo':>8}{'vs demo':>8}{'n':>6}"
print(hdr); print("-"*len(hdr))
for r in flat:
    demo = f"{r.get('demo_base_mae',float('nan')):>8.2f}" if "demo_base_mae" in r else f"{'-':>8}"
    vsd  = f"{r.get('beats_demo',float('nan')):>+8.2f}" if "beats_demo" in r else f"{'-':>8}"
    flag = "" if r["beats_base"]>0 else "   <- fails predict-mean"
    print(f"{r['target']:<12}{r['win_mae']:>9.2f}{r['subj_mae']:>10.2f}{r['within_mae']:>9.2f}"
          f"{r['r']:>7.3f}{r['baseline_mae']:>8.2f}{r['beats_base']:>+8.2f}{demo}{vsd}{r['n_test']:>6}{flag}")

with open("mcd_triage_results.json","w") as f:
    json.dump([r for r in flat], f, indent=2, default=float)
print("\n✓ saved -> mcd_triage_results.json")

target        win MAE  subj MAE   within      r    base   beats    demo vs demo     n
-------------------------------------------------------------------------------------
diastolic_bp     7.49      6.92     1.82 -0.002    7.58   +0.09    7.37   -0.12  1842
systolic_bp     16.18     14.95     3.75  0.066   16.23   +0.04   15.08   -1.10  1842
SpO2             1.10      0.95     0.28  0.042    1.12   +0.02       -       -  1842
RR               1.27      1.08     0.61 -0.017    1.25   -0.02       -       -  1842   <- fails predict-mean
Stiffness        2.17      1.96     0.59 -0.014    2.14   -0.03       -       -  1798   <- fails predict-mean

✓ saved -> mcd_triage_results.json
